# Quickstart: Composable Preprocessing with PreToolkit

This notebook demonstrates end-to-end data preparation using **Preprocessing Toolkit**:

1. **MissingValueImputer** – filling missing values with column-aware statistics.
2. **CategoricalEncoder** – converting categories into one-hot or ordinal representations.
3. **FeatureScaler** – scaling numeric features to zero mean and unit variance.
4. **Pipeline** – chaining blocks cleanly with scikit-learn compatibility.
5. **Data Leakage Protection** – learning strictly on the training set to prevent optimistic leakage.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline

from preprocessing_toolkit import (
    CategoricalEncoder,
    FeatureScaler,
    MissingValueImputer,
    Pipeline,
)

## 1. Create a Synthetic Tabular Dataset

We generate a self-contained synthetic dataset representing customer subscription data. It includes numeric features (`age`, `income`), categorical features (`city`, `tier`), and missing values.

In [2]:
rng = np.random.RandomState(42)
n_samples = 120

age = rng.normal(loc=35, scale=10, size=n_samples).round(1)
income = rng.normal(loc=60000, scale=15000, size=n_samples).round(-2)
city = rng.choice(["New York", "London", "Tokyo", "Berlin"], size=n_samples)
tier = rng.choice(["Bronze", "Silver", "Gold"], size=n_samples)

# Target label: customer purchased subscription
logit = 0.05 * (age - 35) + 0.00005 * (income - 60000)
prob = 1 / (1 + np.exp(-logit))
purchased = (rng.uniform(size=n_samples) < prob).astype(int)

df = pd.DataFrame(
    {
        "age": age,
        "income": income,
        "city": city,
        "tier": tier,
    }
)

# Introduce missing values across numeric and categorical columns
mask_age = rng.uniform(size=n_samples) < 0.10
mask_income = rng.uniform(size=n_samples) < 0.08
mask_city = rng.uniform(size=n_samples) < 0.05

df.loc[mask_age, "age"] = np.nan
df.loc[mask_income, "income"] = np.nan
df.loc[mask_city, "city"] = np.nan

print("Dataset preview:")
print(df.head())
print("\nMissing values per column:")
print(df.isna().sum())

Dataset preview:
    age   income      city    tier
0  40.0  71900.0    Berlin    Gold
1  33.6  46400.0    London  Silver
2  41.5  81000.0     Tokyo  Silver
3  50.2  39000.0    London  Bronze
4  32.7  68800.0  New York    Gold

Missing values per column:
age       10
income     5
city       9
tier       0
dtype: int64


## 2. Train / Test Split

To prevent data leakage, preprocessing statistics must always be learned from the training set only.

In [3]:
X = df
y = purchased

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

Training samples: 90
Test samples:     30


## 3. Running Each Block on Its Own

Before chaining blocks in a pipeline, let's explore what each block does individually.

### 3.1 MissingValueImputer
Fills missing values using statistics learned from the training set:
- Numeric columns use `mean`, `median` (default: `mean`), or `most_frequent`.
- Categorical columns always use their mode (`most_frequent`).

In [4]:
imputer = MissingValueImputer(strategy="median")

# Fit on training data, transform both
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Learned fill values (imputer.statistics_):")
for col, stat in imputer.statistics_.items():
    print(f"  {col}: {stat}")

print(f"\nRemaining missing values in train: {X_train_imp.isna().sum().sum()}")
print(f"Remaining missing values in test:  {X_test_imp.isna().sum().sum()}")

Learned fill values (imputer.statistics_):
  age: 33.6
  income: 62700.0
  city: London
  tier: Bronze

Remaining missing values in train: 0
Remaining missing values in test:  0


### 3.2 CategoricalEncoder
Converts non-numeric (object, string, category) columns to numeric representations:
- `method="onehot"` creates binary dummy columns `<col>_<category>`.
- `method="ordinal"` maps categories to integer ranks.
- `handle_unknown="ignore"` ensures unseen categories in test data do not crash inference.

In [5]:
encoder = CategoricalEncoder(method="onehot", handle_unknown="ignore")

X_train_enc = encoder.fit_transform(X_train_imp)
X_test_enc = encoder.transform(X_test_imp)

print("Learned categories (encoder.categories_):")
for col, cats in encoder.categories_.items():
    print(f"  {col}: {cats}")

print("\nEncoded columns:")
print(list(X_train_enc.columns))

Learned categories (encoder.categories_):
  city: ['Berlin', 'London', 'New York', 'Tokyo']
  tier: ['Bronze', 'Gold', 'Silver']

Encoded columns:
['age', 'income', 'city_Berlin', 'city_London', 'city_New York', 'city_Tokyo', 'tier_Bronze', 'tier_Gold', 'tier_Silver']


### 3.3 FeatureScaler
Scales numeric columns as `(x - center_) / scale_`:
- `method="standard"` subtracts mean and divides by standard deviation.
- `method="minmax"` scales values into `[0, 1]`.
- Non-numeric columns pass through untouched.

In [6]:
scaler = FeatureScaler(method="standard")

X_train_scl = scaler.fit_transform(X_train_enc)
X_test_scl = scaler.transform(X_test_enc)

print("Learned center (mean):")
print({col: round(val, 2) for col, val in list(scaler.center_.items())[:4]})
print("\nLearned scale (std):")
print({col: round(val, 2) for col, val in list(scaler.scale_.items())[:4]})

Learned center (mean):
{'age': 33.76, 'income': 60014.44, 'city_Berlin': 0.24, 'city_London': 0.33}

Learned scale (std):
{'age': 8.37, 'income': 13658.98, 'city_Berlin': 0.43, 'city_London': 0.47}


## 4. Chaining Blocks into a Pipeline

`Pipeline` composes blocks so that a single `fit_transform` or `transform` call executes the entire sequence in order.

In [7]:
pipe = Pipeline(
    [
        ("imputer", MissingValueImputer(strategy="median")),
        ("encoder", CategoricalEncoder(method="onehot", handle_unknown="ignore")),
        ("scaler", FeatureScaler(method="standard")),
    ]
)

# Run the complete preprocessing chain
X_train_clean = pipe.fit_transform(X_train)
X_test_clean = pipe.transform(X_test)

print("Output feature names after full pipeline:")
print(pipe.get_feature_names_out())

print("\nTransformed training sample:")
print(X_train_clean.head())

Output feature names after full pipeline:
['age', 'income', 'city_Berlin', 'city_London', 'city_New York', 'city_Tokyo', 'tier_Bronze', 'tier_Gold', 'tier_Silver']

Transformed training sample:
         age    income  city_Berlin  city_London  city_New York  city_Tokyo  \
24 -0.496753  0.284469     1.758098    -0.707107      -0.568796   -0.464991   
78  0.255541  0.196615    -0.568796     1.414214      -0.568796   -0.464991   
22  0.231659 -1.765465     1.758098    -0.707107      -0.568796   -0.464991   
96  0.506306 -0.850316    -0.568796    -0.707107      -0.568796    2.150581   
56 -0.854989  0.013585     1.758098    -0.707107      -0.568796   -0.464991   

    tier_Bronze  tier_Gold  tier_Silver  
24     1.253566  -0.620174    -0.707107  
78     1.253566  -0.620174    -0.707107  
22     1.253566  -0.620174    -0.707107  
96    -0.797724  -0.620174     1.414214  
56    -0.797724   1.612452    -0.707107  


## 5. scikit-learn Integration & Cross-Validation

`Pipeline` implements scikit-learn's estimator protocol (`fit`, `transform`, `get_params`, `set_params`, `__sklearn_tags__`). It works seamlessly inside scikit-learn pipelines and model selection tools such as `cross_val_score`.

In [8]:
# Compose PreToolkit's Pipeline with a scikit-learn classifier
model_pipeline = make_pipeline(pipe, LogisticRegression(random_state=42))

cv_scores = cross_val_score(model_pipeline, X, y, cv=5, scoring="accuracy")

print(f"5-Fold CV Accuracy: {cv_scores.mean():.3f} (± {cv_scores.std():.3f})")
print(f"Fold scores: {cv_scores.round(3)}")

5-Fold CV Accuracy: 0.542 (± 0.079)
Fold scores: [0.542 0.667 0.542 0.417 0.542]


## 6. Why Fitting on Train Only Matters (Data Leakage)

**Data leakage** occurs when information from outside the training dataset is used to train or preprocess the model. If you compute mean or scaling parameters on the full dataset (or on the test set), the model indirectly accesses test data distribution.

Here we compare the learned training statistics with the actual test set statistics:

In [9]:
learned_train_mean = pipe.named_steps["scaler"].center_["age"]
actual_test_mean = X_test["age"].mean()
transformed_test_mean = X_test_clean["age"].mean()

print(f"Training mean learned during fit():     {learned_train_mean:.2f}")
print(f"Actual mean of test set (unseen):      {actual_test_mean:.2f}")
print(f"Transformed test set mean:              {transformed_test_mean:.4f}")

# The transformed test set mean is NOT 0.0 because it was standardized
# using the training mean, preserving the integrity of the test set:
assert abs(transformed_test_mean) > 1e-3, "Test set mean should reflect training center"
print("\n✓ No data leakage: test data was transformed using training statistics only.")

Training mean learned during fit():     33.76
Actual mean of test set (unseen):      36.85
Transformed test set mean:              0.3300

✓ No data leakage: test data was transformed using training statistics only.
